# Leaders Detection
Détecte les entités `person` (GLiNER checkpoint-800) et les occurrences du mot **Président** (regex)
dans tous les fichiers `.txt` du corpus Archelec, puis construit un tableau annoté
avec contexte (70 caractères avant/après) et métadonnées (année, département, parti).
Une colonne `is_president` indique si l'entité correspond au Président de la République en exercice.

## 1 — Imports & configuration

In [1]:
import os
import re
import unicodedata
from pathlib import Path

import pandas as pd
from gliner import GLiNER
from tqdm.notebook import tqdm

# ── Chemins ───────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.getcwd())          # …/projet_archelec/notebooks
ROOT_DIR     = NOTEBOOK_DIR.parent        # …/projet_archelec

TEXT_DIR     = ROOT_DIR / "data" / "raw" / "arkindex_archelec" / "text_files"
CSV_PATH     = ROOT_DIR / "data" / "raw" / "archelec.csv"
MODEL_PATH   = str((ROOT_DIR / "models" / "Gliner" / "checkpoint-800").resolve())
OUTPUT_DIR   = ROOT_DIR / "data" / "results" / "output_best_model"
OUTPUT_FILE  = OUTPUT_DIR / "leaders_mentions.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Paramètres GLiNER ─────────────────────────────────────────────────────────
TAGS         = ["person"]
THRESHOLD    = 0.8
CHUNK_SIZE   = 512   # augmenté (300 → 512) : moins de chunks → moins de batches
CONTEXT_SIZE = 300

# ── Regex Président (désactivée pour l'instant) ───────────────────────────────
# PRESIDENT_RE = re.compile(r'\bPr[eé]sident[e]?\b', re.IGNORECASE)

# ── Présidents de la République par année d'élection ─────────────────────────
LEADERS = {
    "1973": "Georges Pompidou",
    "1978": "Valéry Giscard d'Estaing",
    "1981": "François Mitterrand",
    "1988": "François Mitterrand",
    "1993": "François Mitterrand",
}

# Noms de famille uniquement — les prénoms seuls ne suffisent pas
LEADER_LASTNAMES = {
    "1973": {"pompidou"},
    "1978": {"giscard", "estaing"},
    "1981": {"mitterrand"},
    "1988": {"mitterrand"},
    "1993": {"mitterrand"},
}

print("Chemins configurés :")
print(f"  TEXT_DIR   : {TEXT_DIR}")
print(f"  CSV_PATH   : {CSV_PATH}")
print(f"  MODEL_PATH : {MODEL_PATH}")
print(f"  OUTPUT     : {OUTPUT_FILE}")


Chemins configurés :
  TEXT_DIR   : /Users/wiamlachqer/Desktop/Named_Entity_Extraction_Archelec_Corpus/projet_archelec/data/raw/arkindex_archelec/text_files
  CSV_PATH   : /Users/wiamlachqer/Desktop/Named_Entity_Extraction_Archelec_Corpus/projet_archelec/data/raw/archelec.csv
  MODEL_PATH : /Users/wiamlachqer/Desktop/Named_Entity_Extraction_Archelec_Corpus/projet_archelec/models/Gliner/checkpoint-800
  OUTPUT     : /Users/wiamlachqer/Desktop/Named_Entity_Extraction_Archelec_Corpus/projet_archelec/data/results/output_best_model/leaders_mentions.xlsx


## 2 — Chargement des métadonnées (archelec.csv)

In [58]:
df_meta = pd.read_csv(CSV_PATH, encoding="latin-1", low_memory=False)
df_meta.columns = df_meta.columns.str.strip()

# Re-encoder latin-1 → UTF-8
for col in df_meta.select_dtypes(include="object").columns:
    df_meta[col] = df_meta[col].apply(
        lambda x: x.encode("latin-1").decode("utf-8") if isinstance(x, str) else x
    )

df_meta["annee"] = df_meta["date"].astype(str).str[:4]

meta_lookup = (
    df_meta
    .set_index("id")[["annee", "departement-nom", "titulaire-soutien"]]
    .rename(columns={"departement-nom": "departement", "titulaire-soutien": "parti"})
    .to_dict(orient="index")
)

print(f"{len(meta_lookup):,} entrées dans la table de correspondance")
sample_key = next(iter(meta_lookup))
print(f"Exemple — {sample_key} : {meta_lookup[sample_key]}")

/var/folders/x5/83lfgsk55sq339swtbb00dk40000gn/T/ipykernel_32974/2840343825.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_meta.select_dtypes(include="object").columns:


33,031 entrées dans la table de correspondance
Exemple — EL009_L_1958_11_001_01_1_PF_01 : {'annee': '1958', 'departement': 'Ain', 'parti': 'Parti radical'}


## 3 — Chargement du modèle GLiNER (checkpoint-800)

In [59]:
model = GLiNER.from_pretrained(
    MODEL_PATH,
    load_tokenizer=True,
    local_files_only=True
)
print("Modèle chargé :", MODEL_PATH)

Modèle chargé : /Users/wiamlachqer/Desktop/Named_Entity_Extraction_Archelec_Corpus/projet_archelec/models/Gliner/checkpoint-800


## 4 — Fonctions utilitaires

In [60]:
import torch

# ── Device : GPU Apple Silicon (MPS) ─────────────────────────────────────────
DEVICE = torch.device("mps")
print(f"Device : {DEVICE}")

model.to(DEVICE)
model.eval()

GLINER_BATCH_SIZE = 128


def normalize(text: str) -> str:
    return unicodedata.normalize("NFD", text.lower()).encode("ascii", "ignore").decode("ascii")


def is_president_match(entity_text: str, annee: str) -> bool:
    lastnames = LEADER_LASTNAMES.get(str(annee))
    if not lastnames:
        return False
    ent_tokens = set(normalize(entity_text).split())
    # Au moins un nom de famille doit être présent (prénom seul → False)
    return bool(ent_tokens & lastnames)


def get_context(text: str, start: int, end: int, window: int = 70) -> str:
    ctx_start = max(0, start - window)
    ctx_end   = min(len(text), end + window)
    return f"{text[ctx_start:start]}[{text[start:end]}]{text[end:ctx_end]}"


def chunk_text(text: str, max_chars: int = CHUNK_SIZE):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        if end < len(text):
            cut = max(
                text.rfind('. ', start, end), text.rfind('!', start, end),
                text.rfind('?', start, end),  text.rfind('\n', start, end),
            )
            if cut > start:
                end = cut + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append((chunk, start))
        start = end
    return chunks


# def extract_president_regex(text: str):
#     return [(m.group(), m.start(), m.end()) for m in PRESIDENT_RE.finditer(text)]


print(f"CHUNK_SIZE={CHUNK_SIZE}  |  GLINER_BATCH_SIZE={GLINER_BATCH_SIZE}")


Device : mps
CHUNK_SIZE=512  |  GLINER_BATCH_SIZE=128


## 5 — Pipeline principal : parcours des fichiers .txt

In [61]:
all_txt_files = sorted(TEXT_DIR.rglob("*.txt"))
print(f"{len(all_txt_files):,} fichiers .txt trouvés\n")

# ── Phase 1 : lecture des fichiers + préparation des chunks ──────────────────
all_chunks      = []   # (chunk_text, char_offset, file_id)
file_meta       = {}   # file_id → (annee, dept, parti, full_text)
skipped_no_meta = 0

print("Phase 1 — lecture des fichiers…")
for txt_path in tqdm(all_txt_files, desc="Lecture"):
    file_id = txt_path.stem
    try:
        text = txt_path.read_text(encoding="utf-8", errors="replace")
    except Exception:
        text = txt_path.read_text(encoding="latin-1", errors="replace")
    text = text.strip()
    if not text:
        continue

    meta = meta_lookup.get(file_id)
    if meta is None:
        parts    = file_id.split("_")
        annee_fb = parts[2] if len(parts) > 2 else "inconnu"
        meta     = {"annee": annee_fb, "departement": "inconnu", "parti": "inconnu"}
        skipped_no_meta += 1

    annee, departement, parti = meta["annee"], meta["departement"], meta["parti"]
    file_meta[file_id] = (annee, departement, parti, text)

    # # ── Regex Président (désactivée) ──────────────────────────────────────
    # for match_text, start, end in extract_president_regex(text):
    #     regex_rows.append({...})

    for chunk, offset in chunk_text(text):
        all_chunks.append((chunk, offset, file_id))

print(f"  → {len(all_chunks):,} chunks préparés (CHUNK_SIZE={CHUNK_SIZE})")
print(f"  → {skipped_no_meta} fichiers sans correspondance CSV")

# ── Phase 2 : inférence GLiNER en batch (only keep is_president=True) ────────
print(f"\nPhase 2 — inférence GLiNER (batch_size={GLINER_BATCH_SIZE})…")
rows = []

with torch.no_grad():
    for i in tqdm(range(0, len(all_chunks), GLINER_BATCH_SIZE), desc="GLiNER batches"):
        batch       = all_chunks[i : i + GLINER_BATCH_SIZE]
        batch_texts = [b[0] for b in batch]
        try:
            batch_results = model.batch_predict_entities(
                batch_texts, TAGS, threshold=THRESHOLD
            )
        except Exception as e:
            print(f"[WARN] batch {i} échoué : {e}")
            batch_results = [[] for _ in batch]

        for entities, (chunk_txt, offset, file_id) in zip(batch_results, batch):
            annee, departement, parti, full_text = file_meta[file_id]
            for ent in entities:
                if not is_president_match(ent["text"], annee):
                    continue   # on ignore les entités non présidentielles
                start = ent["start"] + offset
                end   = ent["end"]   + offset
                rows.append({
                    "annee":            annee,
                    "departement":      departement,
                    "parti":            parti,
                    "file_id":          file_id,
                    "entite":           ent["text"],
                    "actual_president": LEADERS.get(annee, "inconnu"),
                    "text":             get_context(full_text, start, end, window=CONTEXT_SIZE),
                })

print(f"\n{len(rows):,} mentions présidentielles retenues")


21,827 fichiers .txt trouvés

Phase 1 — lecture des fichiers…


Lecture:   0%|          | 0/21827 [00:00<?, ?it/s]

  → 226,628 chunks préparés (CHUNK_SIZE=512)
  → 649 fichiers sans correspondance CSV

Phase 2 — inférence GLiNER (batch_size=128)…


GLiNER batches:   0%|          | 0/1771 [00:00<?, ?it/s]

/var/folders/x5/83lfgsk55sq339swtbb00dk40000gn/T/ipykernel_32974/4110474162.py:49: FutureWarning: GLiNER.batch_predict_entities is deprecated and will be removed in a future release. Please use GLiNER.inference instead.
  batch_results = model.batch_predict_entities(



6,342 mentions présidentielles retenues


## 6 — Construction et aperçu du tableau

In [62]:
df_results = pd.DataFrame(rows, columns=[
    "annee", "departement", "parti", "file_id", "entite", "actual_president", "text"
])

print(f"Dimensions : {df_results.shape}")
print(f"\nRépartition par année :")
print(df_results["annee"].value_counts().sort_index().to_string())
print(f"\nRépartition par président :")
print(df_results["actual_president"].value_counts().to_string())

df_results.head(10)


Dimensions : (6342, 7)

Répartition par année :
annee
1973     343
1978     157
1981    2820
1988    2859
1993     163

Répartition par président :
actual_president
François Mitterrand         5842
Georges Pompidou             343
Valéry Giscard d'Estaing     157


,annee,departement,parti,file_id,entite,actual_president,text
0,1973,Ain,non mentionné,EL065_L_1973_03_001_01_1_PF_05,Georges POMPIDOU,Georges Pompidou,"étiquette, avec la même volonté d'être utile ..."
1,1973,Ain,Républicain indépendant;Union des républicains...,EL065_L_1973_03_001_02_1_PF_01,Georges POMPIDOU,Georges Pompidou,"rages, accompagné de mon fidèle ami M. Michel ..."
2,1973,Aisne,Alliance républicaine indépendante et libérale,EL065_L_1973_03_002_01_1_PF_05,Georges Pompidou,Georges Pompidou,tion de l'Aisne\nGeorges LESIMON Commerçant CI...
3,1973,Aisne,Alliance républicaine indépendante et libérale,EL065_L_1973_03_002_02_1_PF_03,Georges Pompidou,Georges Pompidou,IPOF\n2e circonscription de l'Aisne\nGérard PA...
4,1973,Aisne,Union des républicains de progrès;Union des dé...,EL065_L_1973_03_002_03_1_PF_03,Georges POMPIDOU,Georges Pompidou,"rine.\nAujourd'hui, je suis avec ces référence..."
5,1973,Aisne,Union des républicains de progrès;Union des dé...,EL065_L_1973_03_002_03_1_PF_03,Georges Pompidou,Georges Pompidou,"e voûte de notre régime, c'est l'institu- tion..."
6,1973,Aisne,Alliance républicaine indépendante et libérale,EL065_L_1973_03_002_03_1_PF_04,Georges Pompidou,Georges Pompidou,irconscription de l'Aisne\nFrancis SOULEYREAU\...
7,1973,Aisne,Alliance républicaine indépendante et libérale,EL065_L_1973_03_002_04_1_PF_03,Georges Pompidou,Georges Pompidou,4me circonscription de l'Aisne\nMarie-Anne HER...
8,1973,Aisne,Union des républicains de progrès,EL065_L_1973_03_002_04_1_PF_04,POMPIDOU,Georges Pompidou,a garantie de l'emploi est assurée mais l'AVEN...
9,1973,Alpes-Maritimes,Union des républicains de progrès,EL065_L_1973_03_006_01_1_PF_04,GEORGES POMPIDOU,Georges Pompidou,"t dans l'ordre. « L'intérêt général, aujourd'h..."


## 7 — Classification du sentiment (Anthropic)
Classifie les 200 premières lignes selon si le candidat s'associe positivement au président ou le critique négativement.

In [2]:
df_results = pd.read_excel(OUTPUT_FILE)

In [4]:
import anthropic
from dotenv import load_dotenv

# Charge la clé depuis .env
load_dotenv(ROOT_DIR / ".env")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_KEY")
if not ANTHROPIC_KEY:
    raise ValueError("ANTHROPIC_KEY introuvable dans .env")

client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

def classify_sentiment(context: str, president: str) -> str:
    prompt = (
        f"""Extrait d'une profession de foi française mentionnant {president} :


        {context}


        Le candidat s'associe-t-il positivement au président (soutien, éloge) 
        ou le critique-t-il négativement ? Réponds uniquement : positif ou négatif."""
    )
    msg = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=10,
        messages=[{"role": "user", "content": prompt}],
    )
    answer = msg.content[0].text.strip().lower()
    if "positif" in answer:
        return "positif"
    if "négatif" in answer or "negatif" in answer:
        return "négatif"
    return answer

# Classification des 200 premières lignes uniquement
sentiments = []
for _, row in tqdm(df_results.head(200).iterrows(), total=200, desc="Classification Anthropic"):
    sentiments.append(classify_sentiment(row["text"], row["actual_president"]))

# Colonne sentiment_president : remplie pour les 200 premières, vide sinon
df_results["sentiment_president"] = ""
df_results.loc[df_results.index[:200], "sentiment_president"] = sentiments

print("Distribution des sentiments (200 premières lignes) :")
print(df_results["sentiment_president"].replace("", pd.NA).dropna().value_counts())


Classification Anthropic:   0%|          | 0/200 [00:00<?, ?it/s]

Distribution des sentiments (200 premières lignes) :
sentiment_president
positif    186
négatif     14
Name: count, dtype: int64


## 8 — Sauvegarde en Excel

In [6]:
df_results.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")
print(f"Fichier sauvegardé : {OUTPUT_FILE}")
print(f"Lignes : {len(df_results):,}")

Fichier sauvegardé : /Users/wiamlachqer/Desktop/Named_Entity_Extraction_Archelec_Corpus/projet_archelec/data/results/output_best_model/leaders_mentions.xlsx
Lignes : 6,342
